# 02 — Silver transformations
Standardises key entities, parses timestamps, casts numeric fields, removes duplicate business keys where appropriate, and writes conformed Delta tables.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def trim_string_columns(df):
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    return df

# Orders: one row per order_id
orders = trim_string_columns(spark.table("bronze.orders"))
orders = (orders
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .dropDuplicates(["order_id"]))
orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.orders")

# Customers: one row per customer_id in this source extract
customers = trim_string_columns(spark.table("bronze.customers"))
customers = customers.dropDuplicates(["customer_id"])
customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.customers")

# Products
products = trim_string_columns(spark.table("bronze.products"))
products = products.dropDuplicates(["product_id"])
products = (products
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("double")))
products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.products")

# Sellers
sellers = trim_string_columns(spark.table("bronze.sellers")).dropDuplicates(["seller_id"])
sellers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.sellers")

# Order items
items = trim_string_columns(spark.table("bronze.order_items"))
items = (items
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .withColumn("price", F.col("price").cast("decimal(18,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(18,2)"))
    .dropDuplicates(["order_id", "order_item_id"]))
items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.order_items")

# Payments
payments = trim_string_columns(spark.table("bronze.order_payments"))
payments = (payments
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(18,2)"))
    .dropDuplicates(["order_id", "payment_sequential"]))
payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.order_payments")

# Reviews: source can contain multiple records per order; retain all distinct review rows
reviews = trim_string_columns(spark.table("bronze.order_reviews")).dropDuplicates()
reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.order_reviews")

# Geolocation: multiple records per zip prefix are expected; deduplicate exact rows only
geo = trim_string_columns(spark.table("bronze.geolocation")).dropDuplicates()
geo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.geolocation")

print("Silver transformations completed.")

## Silver row-count summary

In [ ]:
silver_tables = ["orders", "customers", "products", "sellers", "order_items", "order_payments", "order_reviews", "geolocation"]
display(spark.createDataFrame([(t, spark.table(f"silver.{t}").count()) for t in silver_tables], ["table_name", "row_count"]))